<a href="https://colab.research.google.com/github/RealHotnCold/MLA0204-192511036/blob/main/11_Credit_Score_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Credit Score Classification
This notebook demonstrates a complete workflow for classifying credit scores (Poor, Standard, Good) using synthetic data, feature preprocessing, and a Random Forest Classifier.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seed for reproducibility
np.random.seed(42)

# Generate synthetic data
num_samples = 1000

data = {
    'Age': np.random.randint(18, 70, size=num_samples),
    'Annual_Income': np.random.randint(20000, 150000, size=num_samples),
    'Num_Bank_Accounts': np.random.randint(1, 10, size=num_samples),
    'Num_Credit_Card': np.random.randint(1, 12, size=num_samples),
    'Interest_Rate': np.random.randint(1, 35, size=num_samples),
    'Num_of_Loan': np.random.randint(0, 10, size=num_samples),
    'Delay_from_due_date': np.random.randint(0, 40, size=num_samples),
    'Num_of_Delayed_Payment': np.random.randint(0, 25, size=num_samples),
    'Outstanding_Debt': np.random.uniform(100, 5000, size=num_samples),
    'Credit_History_Age_Months': np.random.randint(1, 400, size=num_samples),
    'Monthly_Balance': np.random.uniform(50, 2000, size=num_samples)
}

df = pd.DataFrame(data)

# Formulate a simple rule-based Credit Score label
def assign_credit_score(row):
    score = 100
    score += (row['Annual_Income'] / 1000) * 1.5
    score += row['Credit_History_Age_Months'] * 0.5
    score -= row['Delay_from_due_date'] * 4
    score -= row['Num_of_Delayed_Payment'] * 3
    score -= (row['Outstanding_Debt'] / 100) * 2
    score -= row['Interest_Rate'] * 2

    if score > 200:
        return 'Good'
    elif score > 100:
        return 'Standard'
    else:
        return 'Poor'

df['Credit_Score'] = df.apply(assign_credit_score, axis=1)

# Preview the dataset
display(df.head())

,Age,Annual_Income,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,Outstanding_Debt,Credit_History_Age_Months,Monthly_Balance,Credit_Score
0,56,125186,3,5,13,9,8,16,4525.634385,193,1351.334768,Standard
1,69,54674,3,10,8,1,30,4,3597.160668,150,135.501362,Poor
2,46,55854,8,5,4,7,14,6,4108.913348,131,686.912716,Poor
3,32,66271,8,2,31,0,3,17,3369.926063,102,1265.463146,Poor
4,60,93688,2,5,28,7,39,20,715.352303,310,159.849892,Standard


## Data Preprocessing and Splitting

In [2]:
# Define features (X) and target (y)
X = df.drop(columns=['Credit_Score'])
y = df['Credit_Score']

# Encode categorical target variable ('Poor', 'Standard', 'Good')
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Shape of Train set:", X_train.shape)
print("Shape of Test set:", X_test.shape)

Shape of Train set: (800, 11)
Shape of Test set: (200, 11)


## Model Training and Evaluation

In [3]:
# Initialize and train Random Forest Classifier
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)

# Predictions
y_pred = model.predict(X_test_scaled)

# Evaluation Metrics
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy Score: {accuracy:.4f}\n")

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

Accuracy Score: 0.8950

Classification Report:
              precision    recall  f1-score   support

        Good       0.91      0.82      0.87        51
        Poor       0.93      0.98      0.95        82
    Standard       0.84      0.85      0.84        67

    accuracy                           0.90       200
   macro avg       0.89      0.88      0.89       200
weighted avg       0.90      0.90      0.89       200

